# 3.1 Range Weighting Function

**Learning objectives:**
- Understand range resolution (cτ/2) for rectangular pulses
- Derive range weighting function |W(r)|² from pulse envelope p(t)
- Comprehend matched filter principle and its effect on resolution
- Plot range weighting function for WSR-88D parameters

## Theory: Pulse Geometry

### Rectangular Pulse

A radar transmits a short electromagnetic pulse $p(t)$ that propagates at the speed of light $c$. The pulse has duration $\tau$. Two scatterers at ranges $r_1$ and $r_2$ can be resolved only if they are separated by more than half the pulse length:

$$r_2 - r_1 > \frac{c\tau}{2}$$

If $|r_2 - r_1| < c\tau/2$, the returned pulses from both scatterers overlap and cannot be distinguished. This sets the **range resolution depth**:

$$\Delta r = \frac{c\tau}{2}$$

### Range Weighting Function

The range weighting function $|W(r - r_0)|^2$ describes how the radar weights returns from different ranges. For a rectangular pulse, it is derived from the pulse envelope:

$$|W(r - r_0)|^2 = \left|p\left(\frac{2(r-r_0)}{c}\right)\right|^2 \tag{3.1}$$

This is the **mirror image** of the pulse envelope mapped to range coordinates. The factor of 2 accounts for the two-way travel time (radar sends pulse out and receives echo).

### Figure 3.1 Description

Figure 3.1 illustrates two overlapping rectangular pulses from scatterers at ranges $r_1$ and $r_2$. When the separation is less than $c\tau/2$, the pulses overlap in time, and the radar cannot resolve the individual scatterers. The shading shows the overlap region.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def plot_range_weighting(tau_us=1.0):
    """Plot range weighting function for given pulse width tau (in microseconds)."""
    c = 3e8  # speed of light m/s
    tau = tau_us * 1e-6  # convert to seconds
    r_resolution = c * tau / 2  # range resolution in meters

    # Create range axis
    r = np.linspace(-500, 500, 1000)  # meters
    # Rectangular pulse weighting
    W = np.where(np.abs(r) <= r_resolution, 1.0, 0.0)
    W_sq = W ** 2

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(r, W_sq, 'b-', linewidth=2, label=r'$|W(r)|²$ (rectangular)')
    ax.axvline(x=r_resolution, color='r', linestyle='--', label=f'r₆ = {r_resolution:.0f} m')
    ax.axvline(x=-r_resolution, color='r', linestyle='--')
    ax.fill_between(r, W_sq, alpha=0.2)
    ax.set_xlabel('Range relative to r₀ (m)')
    ax.set_ylabel(r'$|W(r)|²$')
    ax.set_title(f'Range Weighting Function: τ = {tau_us} μs → cτ/2 = {r_resolution:.0f} m')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_range_weighting,
         tau_us=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1,
                            description=r'τ (μs):'))

## Theory: Matched Filter

### Digital Receiver and Convolution

The radar receiver performs a convolution of the received signal with the matched filter impulse response $h(t)$:

$$y(t_i) = \sum_m h(t_i - t_m) \, p(t_m) \tag{3.2}$$

where $p(t_m)$ is the discretized pulse envelope.

### Mirror Image Interpretation

The range weighting function is the **mirror image** of the receiver output:

$$W(r_i) = y\left(-\frac{c t_i}{2}\right) \tag{3.3}$$

This reversal maps time (two-way travel) to range.

### Matched Filter Effect

The **matched filter** uses $h(t)$ equal to the mirror image of $p(t)$:

$$h(t) = p(-t)$$

For a **rectangular pulse**, the matched filter output is a **triangular** weighting function. The convolution of two rectangles (each of width $c\tau/2$) produces a triangle with:

- Base width: $c\tau$
- Peak at center
- Area = 2/3 of the equivalent rectangle

The **range weighting loss** $l_r = 1.5$ (equivalent to 1.76 dB) represents the power loss compared to an ideal rectangular weighting.

### Figure 3.2 Description

Figure 3.2 shows the range weighting function $|W(r)|^2$ on a dB scale for a matched filter. The triangle shape has steeper slopes than the rectangular pulse, resulting in better range sidelobe suppression. The -6 dB width $r_6$ is the standard resolution measure.

In [ ]:
from ipywidgets import interact, FloatSlider

def plot_matched_filter_effect(tau_us=1.0):
    """Show how matched filter changes rectangular p(t) into triangular W(r)."""
    c = 3e8
    tau = tau_us * 1e-6
    r_res = c * tau / 2

    r = np.linspace(-600, 600, 1200)
    # Rectangular pulse envelope
    p = np.where(np.abs(r) <= r_res, 1.0, 0.0)
    # Matched filter: convolution of rectangle with itself = triangle
    from scipy.signal import convolve
    W_triangle = convolve(p, p, mode='same')
    W_triangle = W_triangle / W_triangle.max()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(r, p, 'b-', linewidth=2, label='p(t) rectangular')
    ax1.set_title('Transmitted Pulse p(t)')
    ax1.set_xlabel('Time / Range')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(r, W_triangle**2, 'r-', linewidth=2, label=r'$|W(r)|²$ (matched filter)')
    ax2.set_title('Range Weighting Function |W(r)|² (Matched Filter)')
    ax2.axvline(x=r_res, color='k', linestyle='--', alpha=0.5, label=f'r₆ = {r_res:.0f} m')
    ax2.axvline(x=-r_res, color='k', linestyle='--', alpha=0.5)
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_matched_filter_effect,
         tau_us=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1,
                            description=r'τ (μs):'))

In [ ]:
def plot_wsr88d_example():
    """Replicate Fig. 3.2-style plot with WSR-88D KOUN parameters."""
    # Real parameters from ch3.md
    r_6 = 240  # meters (6 dB width)
    sample_spacing = 50  # meters (research)

    r = np.arange(-400, 401, sample_spacing)
    # Approximate asymmetric range weighting function shape
    # Use different width factors for left/right to create asymmetry
    r_pos = r[r >= 0]
    r_neg = r[r < 0]
    W_pos = np.exp(-4 * np.log(2) * (r_pos / (r_6 * 0.6))**2)  # steeper on right
    W_neg = np.exp(-4 * np.log(2) * (r_neg / (r_6 * 0.8))**2)  # shallower on left
    W_sq = np.concatenate([W_neg[::-1], W_pos])
    W_sq_dB = 10 * np.log10(W_sq + 1e-10)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(r, W_sq_dB, 'b-', linewidth=1.5)
    ax.scatter(r[::2], W_sq_dB[::2], c='red', s=20, zorder=5, label='samples (50 m)')
    ax.axhline(y=-6, color='gray', linestyle='--', alpha=0.7, label='-6 dB (r₆)')
    ax.axhline(y=-40, color='gray', linestyle=':', alpha=0.7, label='-40 dB')
    ax.axvline(x=-r_6/2, color='red', linestyle='--', alpha=0.5)
    ax.axvline(x=r_6/2, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Range relative to center (m)')
    ax.set_ylabel('|W(r)|² (dB)')
    ax.set_title('WSR-88D (KOUN) Range Weighting Function — Fig. 3.2 style')
    ax.legend()
    ax.set_ylim(-60, 5)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_wsr88d_example()

## Summary

Key takeaways from this notebook:

1. **Range resolution** is determined by the pulse width: $\Delta r = c\tau/2$
2. **Range weighting function** $|W(r)|^2$ describes how the radar weights returns from different ranges
3. **Matched filter** uses $h(t) = p(-t)$ to maximize signal-to-noise ratio
4. For **rectangular pulses**, the matched filter produces a **triangular** range weighting function
5. **Range weighting loss** $l_r = 1.5$ (1.76 dB) is inherent to the matched filter approach
6. **WSR-88D** has asymmetric weighting with ~400 m width at -60 dB and $r_6 = 240$ m

## References

**References:**
- Doviak & Zrnic (2006) Doppler Radar and Weather Observations, Sect. 4.4
- Ryzhkov & Zrnic (2024) Radar Polarimetry for Weather Observations, Ch. 3